# ================================================
# Notebook 03 — Feature Engineering & Engagement Scoring
# Project: Ad Engagement Analysis
# Goal: Build a custom business metric to rank ad effectiveness
# ================================================

In this notebook, we move beyond just counting interactions. We are going to build a custom **Engagement Effectiveness Score**.

### The Business Logic:
Not all interactions are equal.
- Base Comment = **1 point** (Minimum effort)
- Emoji Used = **+2 points** (Shows active emotional reaction)
- Hashtags Used = **+1 point per hashtag** (Shows intent to amplify/share the ad visually)

Combining these gives us a weighted score, allowing us to find the campaigns that truly performed the best!

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

# Load and clean the dataset
df = pd.read_csv('../data/comments_cleaned.csv')

if 'Unnamed: 0' in df.columns:
    df.drop('Unnamed: 0', axis=1, inplace=True)
if 'id' in df.columns:
    df.drop('id', axis=1, inplace=True)
    
df.rename(columns={
    'User  id': 'user_id',
    'Photo id': 'campaign_id',
    'created Timestamp': 'timestamp',
    'posted date': 'posted_date',
    'emoji used': 'emoji_used',
    'Hashtags used count': 'hashtag_count'
}, inplace=True)

print("Data loaded successfully.")

In [ ]:
# 1. Feature Engineering: The Engagement Score
# Create the individual components
df['base_comment_score'] = 1
df['emoji_score'] = df['emoji_used'].apply(lambda x: 2 if x == 'yes' else 0)
# hashtag_count is already an integer from 0 to 6

# Calculate Total Score
df['engagement_score'] = df['base_comment_score'] + df['emoji_score'] + df['hashtag_count']

# Let's see some extremely high vs low engagement comments
print("Sample of scored interactions:")
df[['campaign_id', 'comment', 'emoji_used', 'hashtag_count', 'engagement_score']].sample(5)

In [ ]:
# 2. Campaign Level Performance
# Let's aggregate these individual scores to the campaign level!

campaign_performance = df.groupby('campaign_id').agg(
    total_interactions=('user_id', 'count'),
    total_engagement_score=('engagement_score', 'sum'),
    avg_engagement_score=('engagement_score', 'mean')
).reset_index()

# Sort to find the highest-performing campaigns based on true engagement
top_campaigns = campaign_performance.sort_values(by='total_engagement_score', ascending=False).head(10)

plt.figure(figsize=(12, 6))
# Create a bar plot using the new custom metric
sns.barplot(
    data=top_campaigns, 
    x=top_campaigns['campaign_id'].astype(str), 
    y='total_engagement_score', 
    palette='viridis'
)
plt.title('Top 10 Ad Campaigns by Custom Engagement Score', fontsize=14, fontweight='bold')
plt.xlabel('Campaign ID')
plt.ylabel('Total Engagement Score')
plt.show()

In [ ]:
# 3. Save Final Analytical Data to CSV
# We will need these enriched data files to power the Streamlit Dashboard (Day 5)

# Save the user-interaction level data with scores
df.to_csv('../data/ad_interactions_scored.csv', index=False)

# Save the campaign-level aggregated performance metrics
campaign_performance.to_csv('../data/campaign_performance.csv', index=False)

print("✅ Feature Engineered datasets successfully saved to /data folder!")